# Bài tập 3 — Luật kết hợp (Association Rules)

## Bộ dữ liệu D2: Instacart Market Basket Analysis

### Mục tiêu

Khai phá các luật kết hợp trong dữ liệu mua hàng của Instacart nhằm tìm ra:

- Các nhóm hàng (`aisle`) thường xuất hiện cùng nhau trong một đơn hàng.
- Mối liên hệ giữa thời điểm đặt hàng và nhóm hàng được mua.
- Các luật có mức độ liên kết đáng chú ý dựa trên Support, Confidence và Lift.

Quy trình thực hiện:

1. Chuẩn bị dữ liệu giao dịch.
2. Rời rạc hóa và giảm không gian item.
3. Nhị phân hóa dữ liệu giao dịch.
4. Khai phá frequent itemsets bằng Apriori và FP-Growth.
5. Sinh luật kết hợp.
6. Đánh giá bằng Support, Confidence và Lift.
7. Lọc các luật tầm thường, không phù hợp hoặc dư thừa.
8. Phân tích một số luật nổi bật từ kết quả thực tế.

> **Để giảm thời gian và tài nguyên tính toán, bài thực nghiệm sử dụng 200.000 dòng đầu tiên của order_products__prior.csv. Vì vậy, các kết quả frequent itemsets và association rules được hiểu là kết quả trên mẫu dữ liệu này, không đại diện cho toàn bộ dữ liệu Instacart.**

> **Notebook này sử dụng dữ liệu đầu vào là `order_details_clean.csv` — kết quả đã được khảo sát, kiểm tra thiếu/khóa liên kết và làm sạch ở Bài 1 (`khao-sat.ipynb`). Bài 3 không đọc lại dữ liệu thô (`orders.csv`, `order_products__prior.csv`, `products.csv`, `aisles.csv`) và không lặp lại các bước kiểm tra đã thực hiện ở Bài 1; toàn bộ phần bên dưới chỉ tập trung vào bước biến đổi đặc thù của luật kết hợp (gộp `aisle`, rời rạc hóa thời gian, nhị phân hóa) và khai phá luật.**

# 1. Chuẩn bị dữ liệu giao dịch

Ở Bài 1 (`khao-sat.ipynb`), dữ liệu thô Instacart:

- `orders.csv`: thông tin đơn hàng.
- `order_products__prior.csv`: sản phẩm thuộc các đơn hàng trước đó.
- `products.csv`: thông tin sản phẩm.
- `aisles.csv`: nhóm hàng của sản phẩm.

đã được hợp nhất, kiểm tra thiếu/khóa liên kết và làm sạch thành một bảng chi tiết duy nhất **`order_details_clean.csv`** (mức `order_id` × `product_id`, đã có sẵn `aisle`, `order_dow`, `order_hour_of_day`...). Bài 3 đọc trực tiếp file này làm điểm xuất phát.

Trong bài toán luật kết hợp, cần chuyển dữ liệu này về dạng:

`transaction → {item1, item2, item3, ...}`

Trong đó mỗi transaction tương ứng với một đơn hàng.


In [1]:
# ĐỌC DỮ LIỆU ĐÃ LÀM SẠCH

import pandas as pd
import numpy as np
import time

from pathlib import Path
from IPython.display import display

# Tìm thư mục gốc của project (đồng bộ cách tìm ROOT với khao-sat.ipynb)
current = Path.cwd()

while current != current.parent:
    if (current / "data" / "processed" / "D2_instacart").exists():
        break
    current = current.parent

ROOT = current
PROCESSED = ROOT / "data" / "processed" / "D2_instacart"

# order_details_clean.csv: kết quả hợp nhất order_products + orders + products + aisles,
# đã loại bản ghi thiếu product_name/aisle và loại trùng lặp (order_id, product_id) ở Bài 1.
order_details = pd.read_csv(PROCESSED / "order_details_clean.csv")

print(f"order_details: {order_details.shape}")
display(order_details.head())


order_details: (200000, 12)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,product_name,aisle_id,aisle,add_to_cart_order,reordered
0,2,202279,3,5,9,8.0,33120,Organic Egg Whites,86,eggs,1,1
1,2,202279,3,5,9,8.0,28985,Michigan Organic Kale,83,fresh vegetables,2,1
2,2,202279,3,5,9,8.0,9327,Garlic Powder,104,spices seasonings,3,0
3,2,202279,3,5,9,8.0,45918,Coconut Butter,19,oils vinegars,4,1
4,2,202279,3,5,9,8.0,30035,Natural Sweetener,17,baking ingredients,5,0


## 1.1. Định nghĩa transaction / basket

Trong bộ dữ liệu Instacart, `order_id` được chọn làm định danh transaction.

Mỗi `order_id` đại diện cho một đơn hàng, và tất cả `product_id` xuất hiện
trong đơn hàng đó tạo thành một basket.

Biểu diễn:

$$ Transaction = \{product_1, product_2, ..., product_n\} $$

Ví dụ nếu một đơn hàng chứa ba sản phẩm:

$$ Order_1 = \{P_1, P_2, P_3\} $$

thì đây được xem là một transaction có ba item.

Việc sử dụng `order_id` làm transaction phù hợp với bản chất của bài toán
Market Basket Analysis vì các sản phẩm trong cùng một đơn hàng có khả năng
được mua cùng nhau.

In [2]:
transactions = (
    order_details
    .groupby("order_id")["product_id"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="product_basket")
)

transactions["product_count"] = (
    transactions["product_basket"].apply(len)
)

print("=" * 60)
print("TRANSACTION / BASKET")
print("=" * 60)

print(
    f"Số transaction: "
    f"{transactions['order_id'].nunique():,}"
)

print(
    f"Số product khác nhau: "
    f"{order_details['product_id'].nunique():,}"
)

print("\nKích thước basket:")
display(
    transactions["product_count"].describe()
)


TRANSACTION / BASKET
Số transaction: 19,850
Số product khác nhau: 21,494

Kích thước basket:


count    19850.000000
mean        10.075567
std          7.495197
min          1.000000
25%          5.000000
50%          8.000000
75%         14.000000
max         68.000000
Name: product_count, dtype: float64

## 1.2. Gộp `product_id → aisle`

Instacart có số lượng sản phẩm rất lớn. Nếu sử dụng trực tiếp `product_id`,
không gian tìm kiếm itemset của Apriori và FP-Growth sẽ tăng rất nhanh.

Do đó, sản phẩm được chuyển từ mức chi tiết:

`product_id` → `aisle_id` → `aisle`

Mỗi transaction chỉ giữ một `aisle` một lần.

Ví dụ:

`Product A`, `Product B`, `Product C`

nếu cùng thuộc một aisle thì trong basket ở mức nhóm chỉ được biểu diễn là:

`{Aisle X}`


#### **Việc gộp này giúp:**
- Giảm số lượng item.
- Giảm độ thưa của dữ liệu.
- Tăng khả năng xuất hiện đồng thời của các item.
- Làm luật kết hợp dễ diễn giải hơn.

Đây là bước **gộp nhóm/aggregation theo danh mục**, không phải chia một
biến số thành các khoảng.

In [3]:
# GỘP PRODUCT_ID → AISLE
# product_to_aisle được suy trực tiếp từ order_details (đã có sẵn cột "aisle" từ Bài 1),
# không cần đọc lại products.csv / aisles.csv.

product_to_aisle = (
    order_details[["product_id", "aisle"]]
    .drop_duplicates(subset="product_id")
    .set_index("product_id")["aisle"]
    .to_dict()
)


def products_to_aisles(product_list):
    aisle_names = {
        product_to_aisle.get(pid)
        for pid in product_list
    }

    return sorted(
        aisle for aisle in aisle_names
        if aisle is not None
    )


transactions["aisle_basket"] = (
    transactions["product_basket"]
    .apply(products_to_aisles)
)

transactions["aisle_count"] = (
    transactions["aisle_basket"]
    .apply(len)
)

print("=" * 60)
print("GỘP PRODUCT_ID → AISLE")
print("=" * 60)

print(
    f"Số aisle khác nhau: "
    f"{transactions['aisle_basket'].explode().nunique():,}"
)

print("\nKích thước basket sau khi gộp:")
display(
    transactions["aisle_count"].describe()
)


GỘP PRODUCT_ID → AISLE
Số aisle khác nhau: 134

Kích thước basket sau khi gộp:


count    19850.000000
mean         7.267103
std          4.749039
min          1.000000
25%          4.000000
50%          6.000000
75%         10.000000
max         43.000000
Name: aisle_count, dtype: float64

## 1.3. Rời rạc hóa thuộc tính số

Bộ dữ liệu có các thuộc tính số liên quan đến thời gian:

- `order_hour_of_day`: giờ đặt hàng từ 0 đến 23.
- `order_dow`: ngày trong tuần.

Các giá trị số này không thể trực tiếp biểu diễn như một item trong basket.
Vì vậy cần rời rạc hóa thành các nhóm.

### Rời rạc hóa `order_hour_of_day`

Chia thành 4 khoảng:

| Khoảng giờ | Nhãn |
|---|---|
| 00–05 | `Time_Night` |
| 06–11 | `Time_Morning` |
| 12–17 | `Time_Afternoon` |
| 18–23 | `Time_Evening` |

Việc chọn các khoảng trên giúp biến 24 giá trị giờ thành 4 nhóm có ý nghĩa
ngữ cảnh và dễ diễn giải.

### Rời rạc hóa `order_dow`

`order_dow` được chuyển thành:

- `Weekday_Order`: ngày trong tuần.
- `Weekend_Order`: cuối tuần.

Sau rời rạc hóa, các giá trị này có thể được xem như item trong basket.

Ví dụ:

`{fresh vegetables, dairy eggs, Time_Morning, Weekday_Order}`

> **Basket cuối cùng gồm hai loại item: nhóm hàng (aisle) và các item ngữ cảnh được rời rạc hóa từ thời gian đặt hàng (time_period, day_type). Do đó, thuật toán có thể phát hiện cả quan hệ giữa các nhóm hàng và quan hệ giữa ngữ cảnh đặt hàng với nhóm hàng.**

In [4]:
# order_dow / order_hour_of_day là thuộc tính mức đơn hàng, mỗi order_id lặp lại
# trên nhiều dòng (mỗi dòng = 1 sản phẩm) trong order_details -> lấy 1 dòng/đơn hàng.
orders_context = (
    order_details[
        [
            "order_id",
            "order_dow",
            "order_hour_of_day"
        ]
    ]
    .drop_duplicates(subset="order_id")
    .copy()
)


# ------------------------------------------------------------
# Rời rạc hóa giờ đặt hàng
# ------------------------------------------------------------

def classify_time(hour):
    if 0 <= hour < 6:
        return "Time_Night"
    elif 6 <= hour < 12:
        return "Time_Morning"
    elif 12 <= hour < 18:
        return "Time_Afternoon"
    else:
        return "Time_Evening"


orders_context["time_period"] = (
    orders_context["order_hour_of_day"]
    .apply(classify_time)
)


# ------------------------------------------------------------
# Rời rạc hóa ngày trong tuần
# ------------------------------------------------------------

orders_context["day_type"] = np.where(
    orders_context["order_dow"].isin([0, 6]),
    "Weekend_Order",
    "Weekday_Order"
)


print("=" * 60)
print("RỜI RẠC HÓA THUỘC TÍNH SỐ")
print("=" * 60)

print("\nPhân bố thời gian:")
display(
    orders_context["time_period"]
    .value_counts()
    .rename_axis("time_period")
    .reset_index(name="transaction_count")
)

print("\nPhân bố loại ngày:")
display(
    orders_context["day_type"]
    .value_counts()
    .rename_axis("day_type")
    .reset_index(name="transaction_count")
)


RỜI RẠC HÓA THUỘC TÍNH SỐ

Phân bố thời gian:


,time_period,transaction_count
0,Time_Afternoon,9360
1,Time_Morning,6589
2,Time_Evening,3521
3,Time_Night,380



Phân bố loại ngày:


,day_type,transaction_count
0,Weekday_Order,13881
1,Weekend_Order,5969


In [5]:
# GHÉP THÔNG TIN THỜI GIAN VÀO TRANSACTION

transactions = transactions.merge(
    orders_context[
        [
            "order_id",
            "time_period",
            "day_type"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Tạo basket cuối cùng
# ------------------------------------------------------------

transactions["basket"] = transactions.apply(
    lambda row: (
        row["aisle_basket"]
        + [row["time_period"]]
        + [row["day_type"]]
    ),
    axis=1
)

print("=" * 60)
print("BASKET SAU KHI RỜI RẠC HÓA")
print("=" * 60)

display(
    transactions[
        [
            "order_id",
            "aisle_basket",
            "time_period",
            "day_type",
            "basket"
        ]
    ].head(10)
)

BASKET SAU KHI RỜI RẠC HÓA


,order_id,aisle_basket,time_period,day_type,basket
0,2,"[baking ingredients, doughs gelatins bake mixe...",Time_Morning,Weekday_Order,"[baking ingredients, doughs gelatins bake mixe..."
1,3,"[bread, fresh vegetables, packaged vegetables ...",Time_Afternoon,Weekday_Order,"[bread, fresh vegetables, packaged vegetables ..."
2,4,"[breakfast bakery, breakfast bars pastries, ch...",Time_Morning,Weekday_Order,"[breakfast bakery, breakfast bars pastries, ch..."
3,5,"[body lotions soap, candy chocolate, cookies c...",Time_Afternoon,Weekend_Order,"[body lotions soap, candy chocolate, cookies c..."
4,6,"[air fresheners candles, laundry, refrigerated]",Time_Afternoon,Weekday_Order,"[air fresheners candles, laundry, refrigerated..."
5,7,"[frozen produce, refrigerated]",Time_Afternoon,Weekday_Order,"[frozen produce, refrigerated, Time_Afternoon,..."
6,8,[buns rolls],Time_Morning,Weekday_Order,"[buns rolls, Time_Morning, Weekday_Order]"
7,9,"[bread, canned fruit applesauce, cookies cakes...",Time_Evening,Weekend_Order,"[bread, canned fruit applesauce, cookies cakes..."
8,10,"[baby food formula, canned meals beans, cream,...",Time_Morning,Weekend_Order,"[baby food formula, canned meals beans, cream,..."
9,11,"[canned meals beans, chips pretzels, fresh dip...",Time_Evening,Weekday_Order,"[canned meals beans, chips pretzels, fresh dip..."


## 1.4. Nhị phân hóa

Sau khi chuẩn bị basket, dữ liệu có dạng:

$$ Transaction \rightarrow \{item_1,item_2,...,item_n\} $$

Để đưa dữ liệu vào Apriori và FP-Growth, cần chuyển sang ma trận:

$$ Transaction \times Item $$

Sử dụng `TransactionEncoder` của thư viện `mlxtend`.

Quy ước:

- `True`: item xuất hiện trong transaction.
- `False`: item không xuất hiện.

Ví dụ:

| Transaction | dairy eggs | fresh vegetables | Time_Morning |
|---|---:|---:|---:|
| 1 | True | True | False |
| 2 | False | True | True |

Ma trận này là đầu vào cho bước khai phá frequent itemsets.

In [6]:
# NHỊ PHÂN HÓA BẰNG TRANSACTIONENCODER

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

encoded_array = te.fit(
    transactions["basket"]
).transform(
    transactions["basket"]
)

basket_df = pd.DataFrame(
    encoded_array,
    columns=te.columns_,
    index=transactions["order_id"]
)

print("=" * 60)
print("MA TRẬN GIAO DỊCH NHỊ PHÂN")
print("=" * 60)

print(
    f"Số transaction: {basket_df.shape[0]:,}"
)

print(
    f"Số item: {basket_df.shape[1]:,}"
)

print(
    f"Kích thước: {basket_df.shape}"
)

display(basket_df.head())

MA TRẬN GIAO DỊCH NHỊ PHÂN
Số transaction: 19,850
Số item: 140
Kích thước: (19850, 140)


,Time_Afternoon,Time_Evening,Time_Morning,Time_Night,Weekday_Order,Weekend_Order,air fresheners candles,asian foods,baby accessories,baby bath body care,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
order_id,,,,,,,,,,,,,,,,,,,,,
2,False,False,True,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,True,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,False,False,True,False,True,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
5,True,False,False,False,False,True,False,False,False,False,...,True,False,False,False,False,False,False,True,False,False
6,True,False,False,False,True,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 1.5. Output kiểm tra

Kiểm tra dữ liệu sau khi hoàn thành toàn bộ quá trình chuẩn bị:

- Số transaction.
- Số item.
- Kích thước basket.
- Giá trị thiếu.
- Item trùng trong basket.
- Kích thước basket nhỏ nhất/lớn nhất/trung bình.
- Các giá trị xuất hiện trong ma trận.
- Tổng số giá trị `True`.

Dữ liệu chỉ được đưa sang bước khai phá khi ma trận có dạng nhị phân hợp lệ
và không còn giá trị thiếu.

In [7]:
# ============================================================
# 1.5. OUTPUT KIỂM TRA DỮ LIỆU
# ============================================================

print("=" * 70)
print("KIỂM TRA DỮ LIỆU SAU KHI CHUẨN BỊ")
print("=" * 70)


# ------------------------------------------------------------
# 1. Transaction
# ------------------------------------------------------------

n_transactions = (
    transactions["order_id"].nunique()
)


# ------------------------------------------------------------
# 2. Item
# ------------------------------------------------------------

n_items = len(te.columns_)


# ------------------------------------------------------------
# 3. Kích thước matrix
# ------------------------------------------------------------

matrix_rows, matrix_cols = basket_df.shape


# ------------------------------------------------------------
# 4. Missing values
# ------------------------------------------------------------

missing_values = (
    basket_df.isna().sum().sum()
)


# ------------------------------------------------------------
# 5. Item trùng trong basket
# ------------------------------------------------------------

duplicate_items = (
    transactions["basket"]
    .apply(lambda x: len(x) != len(set(x)))
    .sum()
)


# ------------------------------------------------------------
# 6. Basket size
# ------------------------------------------------------------

basket_sizes = basket_df.sum(axis=1)


# ------------------------------------------------------------
# 7. Kiểm tra giá trị matrix
# ------------------------------------------------------------

unique_values = pd.unique(
    basket_df.values.ravel()
)

is_binary = set(
    unique_values
).issubset({True, False})


# ------------------------------------------------------------
# 8. Tổng số item xuất hiện
# ------------------------------------------------------------

total_true = int(
    basket_df.sum().sum()
)


# ============================================================
# HIỂN THỊ KẾT QUẢ
# ============================================================

print("\n--- TỔNG QUAN ---")

print(
    f"Số transaction           : "
    f"{n_transactions:,}"
)

print(
    f"Số item                  : "
    f"{n_items:,}"
)

print(
    f"Kích thước basket matrix : "
    f"({matrix_rows:,}, {matrix_cols:,})"
)

print(
    f"Số giá trị thiếu         : "
    f"{missing_values:,}"
)

print(
    f"Basket có item trùng     : "
    f"{duplicate_items:,}"
)


print("\n--- KÍCH THƯỚC BASKET ---")

print(
    f"Nhỏ nhất                 : "
    f"{basket_sizes.min():.0f}"
)

print(
    f"Lớn nhất                 : "
    f"{basket_sizes.max():.0f}"
)

print(
    f"Trung bình               : "
    f"{basket_sizes.mean():.2f}"
)

print(
    f"Trung vị                 : "
    f"{basket_sizes.median():.2f}"
)


print("\n--- KIỂM TRA MA TRẬN NHỊ PHÂN ---")

print(
    f"Các giá trị xuất hiện   : "
    f"{unique_values}"
)

print(
    f"Tổng số giá trị True    : "
    f"{total_true:,}"
)


# ============================================================
# KIỂM TRA TỔNG THỂ
# ============================================================

print("\n" + "=" * 70)
print("KẾT LUẬN")
print("=" * 70)

if (
    n_transactions > 0
    and n_items > 0
    and missing_values == 0
    and duplicate_items == 0
    and is_binary
):
    print(
        "✓ Dữ liệu hợp lệ."
    )
    print(
        "✓ Basket đã được nhị phân hóa."
    )
    print(
        "✓ Sẵn sàng cho Apriori và FP-Growth."
    )
else:
    print(
        "⚠ Dữ liệu chưa hợp lệ."
    )
    print(
        "⚠ Cần kiểm tra lại trước khi khai phá."
    )

KIỂM TRA DỮ LIỆU SAU KHI CHUẨN BỊ

--- TỔNG QUAN ---
Số transaction           : 19,850
Số item                  : 140
Kích thước basket matrix : (19,850, 140)
Số giá trị thiếu         : 0
Basket có item trùng     : 0

--- KÍCH THƯỚC BASKET ---
Nhỏ nhất                 : 3
Lớn nhất                 : 45
Trung bình               : 9.27
Trung vị                 : 8.00

--- KIỂM TRA MA TRẬN NHỊ PHÂN ---
Các giá trị xuất hiện   : [False  True]
Tổng số giá trị True    : 183,952

KẾT LUẬN
✓ Dữ liệu hợp lệ.
✓ Basket đã được nhị phân hóa.
✓ Sẵn sàng cho Apriori và FP-Growth.


# 2. Thuật toán Apriori và FP-Growth

Sau khi xây dựng ma trận basket dạng nhị phân, tiến hành khai thác
các tập mục phổ biến (frequent itemsets) bằng hai thuật toán:

- **Apriori**
- **FP-Growth**

Hai mức `min_support` được sử dụng:

- `0.01`: ngưỡng cao, giúp giảm số lượng itemset và luật sinh ra.
- `0.005`: ngưỡng thấp hơn, cho phép phát hiện thêm các itemset ít phổ biến hơn.

Việc sử dụng hai mức ngưỡng giúp đánh giá ảnh hưởng của `min_support`
đến số lượng frequent itemsets và thời gian thực thi.

In [8]:
from mlxtend.frequent_patterns import apriori, fpgrowth
import pandas as pd
import time

print("✓ Đã import Apriori và FP-Growth")

SUPPORT_HIGH = 0.01
SUPPORT_LOW = 0.005

MAX_LEN = 3

print("=" * 70)
print("CẤU HÌNH KHAI THÁC FREQUENT ITEMSETS")
print("=" * 70)

print(f"Min support cao : {SUPPORT_HIGH}")
print(f"Min support thấp: {SUPPORT_LOW}")
print(f"Max itemset size: {MAX_LEN}")

✓ Đã import Apriori và FP-Growth
CẤU HÌNH KHAI THÁC FREQUENT ITEMSETS
Min support cao : 0.01
Min support thấp: 0.005
Max itemset size: 3


> **MAX_LEN = 3 được sử dụng để giới hạn kích thước itemset, nhằm kiểm soát không gian tìm kiếm và tránh số lượng tổ hợp tăng quá lớn. Do đó, các luật được phân tích trong bài chỉ dựa trên các frequent itemsets có tối đa 3 item.**

In [9]:
# CHẠY THUẬT TOÁN APRIORI

print("=" * 70)
print("CHẠY THUẬT TOÁN APRIORI")
print("=" * 70)

apriori_results = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    
    print(f"\n--- Apriori | min_support = {support} ---")
    
    start_time = time.perf_counter()
    
    frequent = apriori(
        basket_df,
        min_support=support,
        use_colnames=True,
        max_len=MAX_LEN,
        low_memory=True
    )
    
    elapsed = time.perf_counter() - start_time
    
    apriori_results[support] = {
        "frequent_itemsets": frequent,
        "time": elapsed
    }
    
    print(f"Số frequent itemsets: {len(frequent):,}")
    print(f"Thời gian chạy       : {elapsed:.4f} giây\n")

# CHẠY THUẬT TOÁN FP-GROWTH

print("=" * 70)
print("CHẠY THUẬT TOÁN FP-GROWTH")
print("=" * 70)

fpgrowth_results = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    
    print(f"\n--- FP-Growth | min_support = {support} ---")
    
    start_time = time.perf_counter()
    
    frequent = fpgrowth(
        basket_df,
        min_support=support,
        use_colnames=True,
        max_len=MAX_LEN
    )
    
    elapsed = time.perf_counter() - start_time
    
    fpgrowth_results[support] = {
        "frequent_itemsets": frequent,
        "time": elapsed
    }
    
    print(f"Số frequent itemsets: {len(frequent):,}")
    print(f"Thời gian chạy       : {elapsed:.4f} giây")

CHẠY THUẬT TOÁN APRIORI

--- Apriori | min_support = 0.01 ---
Số frequent itemsets: 3,883
Thời gian chạy       : 0.3755 giây


--- Apriori | min_support = 0.005 ---
Số frequent itemsets: 9,563
Thời gian chạy       : 0.5605 giây

CHẠY THUẬT TOÁN FP-GROWTH

--- FP-Growth | min_support = 0.01 ---
Số frequent itemsets: 3,883
Thời gian chạy       : 1.3189 giây

--- FP-Growth | min_support = 0.005 ---
Số frequent itemsets: 9,563
Thời gian chạy       : 2.4426 giây


In [10]:
print("=" * 70)
print("VÍ DỤ FREQUENT ITEMSETS")
print("=" * 70)

frequent_apriori = apriori_results[SUPPORT_HIGH]["frequent_itemsets"]

display(
    frequent_apriori
    .sort_values("support", ascending=False)
    .head(20)
)

VÍ DỤ FREQUENT ITEMSETS


,support,itemsets
4,0.699295,frozenset({Weekday_Order})
39,0.557128,frozenset({fresh fruits})
0,0.471537,frozenset({Time_Afternoon})
42,0.440151,frozenset({fresh vegetables})
320,0.377935,"frozenset({fresh fruits, Weekday_Order})"
75,0.363526,frozenset({packaged vegetables fruits})
2,0.331940,frozenset({Time_Morning})
102,0.324987,"frozenset({Time_Afternoon, Weekday_Order})"
880,0.316877,"frozenset({fresh fruits, fresh vegetables})"
5,0.300705,frozenset({Weekend_Order})


### So sánh Apriori và FP-Growth

Hai thuật toán được chạy trên cùng một ma trận basket và cùng các mức
`min_support`, do đó có thể so sánh dựa trên:

- Số lượng frequent itemsets tìm được.
- Thời gian thực thi.
- Ảnh hưởng của việc giảm `min_support`.

Về nguyên tắc, Apriori và FP-Growth phải cho cùng tập frequent itemsets
khi sử dụng cùng dữ liệu và cùng ngưỡng `min_support`. Sự khác biệt chủ
yếu nằm ở phương pháp tìm kiếm và thời gian thực thi.

In [11]:
comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    
    apriori_data = apriori_results[support]
    fpgrowth_data = fpgrowth_results[support]
    
    comparison_rows.append({
        "Algorithm": "Apriori",
        "min_support": support,
        "Frequent itemsets": len(apriori_data["frequent_itemsets"]),
        "Runtime (s)": round(apriori_data["time"], 4)
    })
    
    comparison_rows.append({
        "Algorithm": "FP-Growth",
        "min_support": support,
        "Frequent itemsets": len(fpgrowth_data["frequent_itemsets"]),
        "Runtime (s)": round(fpgrowth_data["time"], 4)
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

print("=" * 70)
print("SO SÁNH ẢNH HƯỞNG CỦA MIN_SUPPORT")
print("=" * 70)

for algorithm, results in [
    ("Apriori", apriori_results),
    ("FP-Growth", fpgrowth_results)
]:
    
    n_high = len(results[SUPPORT_HIGH]["frequent_itemsets"])
    n_low = len(results[SUPPORT_LOW]["frequent_itemsets"])
    
    increase = n_low - n_high
    increase_pct = increase / n_high * 100
    
    print(f"\n{algorithm}")
    print(f"  min_support = {SUPPORT_HIGH}: {n_high:,} itemsets")
    print(f"  min_support = {SUPPORT_LOW}: {n_low:,} itemsets")
    print(f"  Tăng thêm             : {increase:,} itemsets")
    print(f"  Tỷ lệ tăng            : {increase_pct:.2f}%")

,Algorithm,min_support,Frequent itemsets,Runtime (s)
0,Apriori,0.010,3883,0.3755
1,FP-Growth,0.010,3883,1.3189
2,Apriori,0.005,9563,0.5605
3,FP-Growth,0.005,9563,2.4426


SO SÁNH ẢNH HƯỞNG CỦA MIN_SUPPORT

Apriori
  min_support = 0.01: 3,883 itemsets
  min_support = 0.005: 9,563 itemsets
  Tăng thêm             : 5,680 itemsets
  Tỷ lệ tăng            : 146.28%

FP-Growth
  min_support = 0.01: 3,883 itemsets
  min_support = 0.005: 9,563 itemsets
  Tăng thêm             : 5,680 itemsets
  Tỷ lệ tăng            : 146.28%


### Nhận xét

Ở `min_support = 0.01`, cả Apriori và FP-Growth đều tìm được
**3,867 frequent itemsets**. Khi giảm ngưỡng xuống `0.005`, số lượng
tăng lên **9,535 frequent itemsets**.

Như vậy, việc giảm `min_support` làm tăng đáng kể không gian tìm kiếm,
cho phép phát hiện thêm các tập mục có mức độ phổ biến thấp hơn.

Về thời gian thực thi trên tập dữ liệu đang sử dụng, Apriori nhanh hơn
FP-Growth:

- Apriori, `min_support = 0.01`: **0.7073 giây**
- Apriori, `min_support = 0.005`: **1.3972 giây**
- FP-Growth, `min_support = 0.01`: **2.9181 giây**
- FP-Growth, `min_support = 0.005`: **5.0047 giây**

Trong thực nghiệm này, Apriori có thời gian chạy thấp hơn FP-Growth.
Đây là kết quả thực nghiệm trên dữ liệu và cấu hình hiện tại, không nên
suy rộng thành kết luận rằng Apriori luôn nhanh hơn FP-Growth trong mọi
trường hợp.

Điểm quan trọng là cả hai thuật toán cho cùng số lượng frequent itemsets
ở cùng một `min_support`, cho thấy kết quả khai thác là nhất quán.

# 3. Độ đo và lựa chọn ngưỡng trong luật kết hợp

Sau khi tìm được các **frequent itemsets**, bước tiếp theo là xây dựng và đánh giá các **association rules (luật kết hợp)**.

Một luật kết hợp có dạng:

$$
A \rightarrow B
$$

Trong đó:

- $A$: tập sản phẩm ở vế trái, gọi là **antecedent**.
- $B$: tập sản phẩm ở vế phải, gọi là **consequent**.
- $A \cap B = \emptyset$.

Ví dụ:

$$
\{banana\} \rightarrow \{milk\}
$$

có thể hiểu là:

> Khi một giao dịch có `banana`, giao dịch đó có xu hướng đồng thời chứa `milk`.

Để đánh giá một luật, ba độ đo quan trọng được sử dụng là:

1. **Support**
2. **Confidence**
3. **Lift**

Ngoài ra, cần lựa chọn các ngưỡng phù hợp để loại bỏ những luật quá yếu hoặc không có ý nghĩa thực tế.

## 3.1. Support
### **Định nghĩa**

**Support** đo mức độ phổ biến của một tập sản phẩm trong toàn bộ tập dữ liệu giao dịch.

Đối với luật:

$$
A \rightarrow B
$$

support được tính dựa trên việc **A và B cùng xuất hiện trong một giao dịch**.

Support càng cao nghĩa là sự kết hợp giữa A và B xuất hiện càng thường xuyên trong dữ liệu.

Ví dụ:

Nếu có 100.000 giao dịch và 2.000 giao dịch chứa đồng thời A và B thì:

$$
Support(A \rightarrow B)
=
\frac{2000}{100000}
=
0.02
$$

hay **2%**.

Điều này có nghĩa là 2% tổng số giao dịch chứa đồng thời A và B.

### **Công thức**

Gọi:

- $N$: tổng số giao dịch.
- $count(A \cup B)$: số giao dịch chứa đồng thời A và B.

Khi đó:

$$
Support(A \rightarrow B)
=
\frac{count(A \cup B)}{N}
$$

Support không phụ thuộc vào hướng của luật.

Do đó:

$$
Support(A \rightarrow B)
=
Support(B \rightarrow A)
$$

Ví dụ:

Nếu:

- Tổng số giao dịch = 10.000.
- Có 300 giao dịch chứa cả `A` và `B`.

thì:

$$
Support(A \rightarrow B)
=
\frac{300}{10000}
=
0.03
$$

hay **3%**.

## 3.2. Confidence
### **Định nghĩa**

**Confidence** đo xác suất xuất hiện của \(B\) khi \(A\) đã xuất hiện.

Nói đơn giản:

> Trong những giao dịch đã mua A, có bao nhiêu phần trăm cũng mua B?

Confidence có tính đến **hướng của luật**.

Do đó:

$$
Confidence(A \rightarrow B)
$$

có thể khác:

$$
Confidence(B \rightarrow A)
$$

Confidence càng cao thì khả năng B xuất hiện khi A xuất hiện càng lớn.

### **Công thức**

Confidence được tính bằng support của A và B chia cho support của A:

$$
Confidence(A \rightarrow B)
=
\frac{Support(A \cup B)}
{Support(A)}
$$

Tương đương:

$$
Confidence(A \rightarrow B)
=
P(B|A)
$$

Ví dụ:

Giả sử:

- 1.000 giao dịch có A.
- 600 trong số đó cũng có B.

Khi đó:

$$
Confidence(A \rightarrow B)
=
\frac{600}{1000}
=
0.6
$$

hay **60%**.

Điều này có nghĩa:

> Trong các giao dịch chứa A, 60% cũng chứa B.

## 3.3. Lift
### **Định nghĩa**

**Lift** đo mức độ liên kết giữa A và B bằng cách so sánh xác suất xuất hiện B khi A xuất hiện với xác suất B xuất hiện một cách độc lập.

Lift giúp khắc phục một hạn chế của confidence.

Ví dụ, nếu sản phẩm B vốn đã xuất hiện trong phần lớn các giao dịch thì nhiều luật:

$$
A \rightarrow B
$$

có thể có confidence cao dù A không thực sự liên quan đến B.

Vì vậy cần sử dụng Lift để kiểm tra xem A và B có thực sự xuất hiện cùng nhau nhiều hơn mức kỳ vọng hay không.

### **Công thức**

Lift được tính:

$$
Lift(A \rightarrow B)
=
\frac{Confidence(A \rightarrow B)}
{Support(B)}
$$

Hoặc:

$$
Lift(A \rightarrow B)
=
\frac{Support(A \cup B)}
{Support(A)\times Support(B)}
$$

### **Ý nghĩa**

**Lift > 1**

A và B có xu hướng xuất hiện cùng nhau nhiều hơn mức kỳ vọng nếu chúng độc lập.

**Lift = 1**

A và B gần như độc lập.

**Lift < 1**

A và B có xu hướng xuất hiện cùng nhau ít hơn mức kỳ vọng.

Do đó, trong khai phá luật kết hợp, các luật có **Lift > 1** thường được quan tâm hơn.

## 3.4. Ví dụ minh họa

Giả sử có 10 giao dịch:

| Transaction | Sản phẩm |
|---|---|
| T1 | A, B |
| T2 | A, B |
| T3 | A, B |
| T4 | A |
| T5 | A |
| T6 | B |
| T7 | B |
| T8 | C |
| T9 | C |
| T10 | A, C |

Xét luật:

$$
A \rightarrow B
$$

Ta có:

- A xuất hiện trong 6 giao dịch: T1, T2, T3, T4, T5, T10.
- B xuất hiện trong 5 giao dịch: T1, T2, T3, T6, T7.
- A và B cùng xuất hiện trong 3 giao dịch: T1, T2, T3.

### **Support**

$$
Support(A \rightarrow B)
=
\frac{3}{10}
=
0.3
$$

### **Confidence**

$$
Confidence(A \rightarrow B)
=
\frac{Support(A\cup B)}
{Support(A)}
=
\frac{0.30}{0.60}
=
0.5
$$

hay **50%**.

### **Lift**

$$
Lift(A \rightarrow B)
=
\frac{0.50}{0.50}
=
1
$$

Như vậy, mặc dù confidence đạt 50%, Lift = 1 cho thấy A và B không thể hiện mối liên hệ mạnh hơn mức kỳ vọng nếu chúng độc lập.

In [12]:
# Ví dụ minh họa cách tính Support, Confidence và Lift

total_transactions = 10

count_A = 6
count_B = 5
count_AB = 3

support_AB = count_AB / total_transactions
support_A = count_A / total_transactions
support_B = count_B / total_transactions

confidence_A_to_B = support_AB / support_A
lift_A_to_B = confidence_A_to_B / support_B

print(f"Support(A → B)    = {support_AB:.2f}")
print(f"Confidence(A → B) = {confidence_A_to_B:.2f}")
print(f"Lift(A → B)       = {lift_A_to_B:.2f}")

Support(A → B)    = 0.30
Confidence(A → B) = 0.50
Lift(A → B)       = 1.00


## 3.5. Lựa chọn ngưỡng

Việc lựa chọn ngưỡng cho association rules cần cân bằng giữa:

- **Không bỏ sót những mối quan hệ hữu ích**.
- **Không tạo ra quá nhiều luật yếu hoặc không có ý nghĩa**.

Nếu chọn ngưỡng quá thấp, số lượng luật có thể tăng lên rất lớn.

Ví dụ:

- Support quá thấp → giữ lại nhiều itemset hiếm.
- Confidence quá thấp → giữ lại các luật có khả năng dự đoán yếu.
- Không kiểm soát Lift → có thể giữ lại những luật có confidence cao chỉ vì consequent vốn đã phổ biến.

Khi đó notebook có thể sinh ra **hàng nghìn hoặc thậm chí hàng chục nghìn luật**, nhưng phần lớn không mang lại thông tin hữu ích.

Do đó, trong bài này sử dụng quy trình:

$$
\text{Frequent Itemsets}
\rightarrow
\text{Association Rules}
\rightarrow
\text{Confidence Filter}
\rightarrow
\text{Lift Filter}
$$

### Ngưỡng được lựa chọn

Ta sử dụng:

$$
Confidence \geq 0.5
$$

và:

$$
Lift \geq 1.2
$$

Tức là chỉ giữ các luật thỏa mãn đồng thời:

- Confidence ít nhất **50%**.
- Lift lớn hơn **1.2**.

### Vì sao chọn Confidence = 0.5?

Confidence 50% nghĩa là:

> Khi A xuất hiện, B cũng xuất hiện trong ít nhất một nửa các trường hợp.

Đây là một mức tương đối nghiêm ngặt để loại bỏ các luật có khả năng dự đoán quá thấp.

Tuy nhiên, confidence một mình chưa đủ vì nó có thể cao do B vốn phổ biến.

### Vì sao chọn Lift $\geq$ 1.2?

Lift $\geq$ 1.2 yêu cầu A và B có xu hướng xuất hiện cùng nhau nhiều hơn mức kỳ vọng khi chúng độc lập.

Vì vậy kết hợp:

$$
Confidence \geq 0.5
$$

và

$$
Lift \geq 1.2
$$

giúp tập trung vào các luật vừa có khả năng xuất hiện B khi có A, vừa thể hiện mối liên hệ dương giữa A và B.

### Lưu ý về Support

Hai ngưỡng support đã được sử dụng ở phần trước:

- `0.01` — 1%.
- `0.005` — 0.5%.

Ta giữ cả hai mức để đánh giá ảnh hưởng của support đến số lượng luật.

Ngưỡng `0.005` thấp hơn giúp phát hiện các mối quan hệ ít phổ biến hơn, nhưng đồng thời tạo ra nhiều luật hơn và cần lọc mạnh hơn bằng confidence và lift.

In [13]:
from mlxtend.frequent_patterns import association_rules

# Sinh toàn bộ association rules từ kết quả Apriori
apriori_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    frequent_itemsets = apriori_results[support]["frequent_itemsets"]

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0
    )

    apriori_rules[support] = rules

CONFIDENCE_THRESHOLD = 0.5
LIFT_THRESHOLD = 1.2

filtered_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    rules = apriori_rules[support]

    filtered = rules[
        (rules["confidence"] >= CONFIDENCE_THRESHOLD) &
        (rules["lift"] >= LIFT_THRESHOLD)
    ].copy()

    filtered_rules[support] = filtered

    print("=" * 70)
    print(f"min_support = {support}")
    print(f"Tổng association rules : {len(rules):,}")
    print(f"Sau khi lọc            : {len(filtered):,}")

min_support = 0.01
Tổng association rules : 18,046
Sau khi lọc            : 1,627
min_support = 0.005
Tổng association rules : 48,338
Sau khi lọc            : 3,341


## 3.6. So sánh ảnh hưởng của ngưỡng

Ta so sánh số lượng luật trước và sau khi áp dụng các ngưỡng:

$$
Confidence \geq 0.5
$$

và:

$$
Lift \geq 1.2
$$

Mục đích là kiểm tra xem việc lọc có giúp giảm đáng kể số lượng luật hay không.

Nếu số luật giảm mạnh sau khi lọc, điều đó cho thấy việc sinh tất cả các luật mà không đặt ngưỡng sẽ tạo ra nhiều luật yếu và khó sử dụng trong thực tế.

In [14]:
comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    total_rules = len(apriori_rules[support])
    filtered_count = len(filtered_rules[support])

    removed = total_rules - filtered_count

    removed_pct = (
        removed / total_rules * 100
        if total_rules > 0 else 0
    )

    comparison_rows.append({
        "min_support": support,
        "Association rules": total_rules,
        "Rules after filtering": filtered_count,
        "Rules removed": removed,
        "Removed (%)": round(removed_pct, 2)
    })

rules_comparison_df = pd.DataFrame(comparison_rows)

display(rules_comparison_df)

,min_support,Association rules,Rules after filtering,Rules removed,Removed (%)
0,0.010,18046,1627,16419,90.98
1,0.005,48338,3341,44997,93.09


## 3.7. So sánh hai mức Support

Hai mức support được sử dụng:

- `min_support = 0.01`
- `min_support = 0.005`

Ở phần trước, kết quả cho thấy:

| min_support | Frequent itemsets |
|---:|---:|
| 0.01 | 3,867 |
| 0.005 | 9,535 |

Khi giảm support từ 0.01 xuống 0.005:

$$
9535 - 3867 = 5668
$$

frequent itemsets được phát hiện thêm.

Tỷ lệ tăng:

$$
\frac{5668}{3867}\times100
\approx146.57\%
$$

Do giảm min_support làm tăng số frequent itemsets, không gian ứng viên để sinh association rules cũng mở rộng, từ đó thường tạo ra nhiều luật hơn.

Tuy nhiên, nhiều luật bổ sung có thể là các mối quan hệ hiếm. Vì vậy cần sử dụng confidence và lift để tiếp tục lọc.

In [15]:
support_comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    support_comparison_rows.append({
        "min_support": support,
        "Frequent itemsets": len(
            apriori_results[support]["frequent_itemsets"]
        ),
        "Association rules": len(
            apriori_rules[support]
        ),
        "Rules after filtering": len(
            filtered_rules[support]
        )
    })

support_comparison_df = pd.DataFrame(
    support_comparison_rows
)

display(support_comparison_df)

,min_support,Frequent itemsets,Association rules,Rules after filtering
0,0.010,3883,18046,1627
1,0.005,9563,48338,3341


# 4. Lọc luật

Sau khi sinh association rules, số lượng luật có thể rất lớn và không phải luật nào cũng có giá trị phân tích. Một số luật có thể yếu, tầm thường hoặc dư thừa.

Do đó, cần áp dụng các tiêu chí lọc dựa trên các độ đo đã trình bày ở phần trước:

- **Support**: loại các luật xuất hiện quá hiếm.
- **Confidence**: loại các luật có khả năng suy diễn thấp.
- **Lift**: loại các luật có mức liên kết gần với độc lập.

Mục tiêu của quá trình lọc là giảm số lượng luật nhưng vẫn giữ lại các luật có mức độ phổ biến và liên kết đủ mạnh để phân tích.

## 4.1. Tiêu chí lọc

Trong bài toán này, các luật được lọc theo hai điều kiện chính:

$$
Confidence(A \rightarrow B) \geq 0.5
$$

và:

$$
Lift(A \rightarrow B) \geq 1.2
$$

### Confidence ≥ 0.5

Điều kiện này yêu cầu ít nhất 50% các giao dịch chứa antecedent cũng chứa consequent.

Confidence thấp thường biểu thị khả năng suy diễn yếu và khó sử dụng cho các quyết định dựa trên luật kết hợp.

### Lift ≥ 1.2

Lift lớn hơn 1 cho thấy A và B có xu hướng xuất hiện cùng nhau nhiều hơn trường hợp độc lập.

Sử dụng ngưỡng 1.2 thay vì chỉ `Lift > 1` giúp loại bỏ các luật có mức liên kết chỉ cao hơn độc lập một cách không đáng kể.

Các ngưỡng trên được xem là **ngưỡng thực nghiệm**, được lựa chọn nhằm cân bằng giữa số lượng luật và mức độ hữu ích của luật.

Nếu đặt ngưỡng quá thấp, hệ thống có thể sinh ra hàng nghìn luật, trong đó nhiều luật yếu hoặc tầm thường, gây khó khăn cho việc phân tích.

In [16]:
CONFIDENCE_THRESHOLD = 0.5
LIFT_THRESHOLD = 1.2

filtered_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    rules = apriori_rules[support]

    filtered = rules[
        (rules["confidence"] >= CONFIDENCE_THRESHOLD) &
        (rules["lift"] >= LIFT_THRESHOLD)
    ].copy()

    filtered_rules[support] = filtered

    print(f"min_support = {support}")
    print(f"Số luật sau lọc: {len(filtered):,}")

min_support = 0.01
Số luật sau lọc: 1,627
min_support = 0.005
Số luật sau lọc: 3,341


## 4.2. Loại luật dư thừa và lựa chọn luật tiêu biểu

Ngoài các tiêu chí định lượng, cần hạn chế các luật cung cấp thông tin tương tự nhau hoặc quá hiển nhiên.

Một cách tiếp cận đơn giản là xếp hạng các luật theo:

1. **Lift giảm dần** – ưu tiên mức độ liên kết mạnh.
2. **Confidence giảm dần** – ưu tiên khả năng suy diễn cao.
3. **Support giảm dần** – ưu tiên luật xuất hiện phổ biến hơn.

Ngoài các tiêu chí định lượng, các luật sau lọc được xếp hạng theo Lift, Confidence và Support để ưu tiên những luật có mức độ liên kết và độ tin cậy cao. Việc chỉ trình bày một số luật đứng đầu giúp giảm độ dài kết quả và tập trung vào các luật đáng chú ý. Đây là bước xếp hạng và lựa chọn để trình bày, không phải một thuật toán loại bỏ dư thừa hoàn toàn.

In [17]:
ranked_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    rules = filtered_rules[support].copy()

    rules = rules.sort_values(
        by=["lift", "confidence", "support"],
        ascending=[False, False, False]
    )

    ranked_rules[support] = rules

def format_itemset(itemset):
    return ", ".join(sorted(itemset))


for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    display_rules = ranked_rules[support].head(20).copy()

    display_rules["antecedents"] = (
        display_rules["antecedents"].apply(format_itemset)
    )

    display_rules["consequents"] = (
        display_rules["consequents"].apply(format_itemset)
    )

    display(
        display_rules[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ]
    )

,antecedents,consequents,support,confidence,lift
17698,"milk, pasta sauce",packaged cheese,0.011184,0.573643,2.476473
13804,"crackers, lunch meat",packaged cheese,0.010882,0.530713,2.291136
17530,"instant foods, packaged vegetables fruits",packaged cheese,0.012443,0.526652,2.273608
10972,"baby food formula, packaged cheese",yogurt,0.010781,0.596100,2.256836
17909,"pasta sauce, yogurt",packaged cheese,0.011285,0.522145,2.254147
17854,"packaged vegetables fruits, pasta sauce",packaged cheese,0.016373,0.515873,2.227073
15538,"fresh fruits, granola",yogurt,0.012544,0.587264,2.223382
13324,"chips pretzels, lunch meat",packaged cheese,0.013703,0.514178,2.219754
16393,fresh herbs,"fresh vegetables, packaged vegetables fruits",0.046348,0.511680,2.215716
11782,"bread, lunch meat",packaged cheese,0.016977,0.508296,2.194360


,antecedents,consequents,support,confidence,lift
32684,"crackers, instant foods",chips pretzels,0.005894,0.504310,3.057593
46462,"instant foods, lunch meat",packaged cheese,0.006348,0.605769,2.615163
45992,"granola, packaged cheese",yogurt,0.006146,0.681564,2.580402
47516,"other creams cheeses, pasta sauce",packaged cheese,0.005592,0.593583,2.562553
46796,"lunch meat, pasta sauce",packaged cheese,0.006700,0.583333,2.518305
45980,"granola, milk",yogurt,0.006398,0.654639,2.478464
47048,"milk, pasta sauce",packaged cheese,0.011184,0.573643,2.476473
36002,"crackers, pasta sauce",packaged cheese,0.005995,0.572115,2.469876
46834,"lunch meat, tortillas flat bread",packaged cheese,0.005793,0.569307,2.457752
37268,"dry pasta, lunch meat",packaged cheese,0.006952,0.567901,2.451683


# 5. Nhận xét kỹ thuật 4 luật nổi bật

Sau khi lọc theo `Confidence ≥ 0.5` và `Lift ≥ 1.2`, các luật được xếp hạng theo `Lift`, `Confidence` và `Support`. Dựa trên kết quả thu được, bốn luật tiêu biểu được lựa chọn dựa trên các tiêu chí khác nhau, bao gồm mức Lift cao, Confidence cao, Support tương đối lớn và khả năng thể hiện quan hệ giữa item sản phẩm với item ngữ cảnh.


### Luật 1: `crackers, instant foods → chips pretzels`

* **Support = 0.005894**
* **Confidence = 0.504310**
* **Lift = 3.057593**

Luật này có **Lift cao nhất** trong các luật được hiển thị. Confidence ≈ 50.4% cho biết trong các giao dịch chứa `crackers` và `instant foods`, khoảng 50.4% đồng thời chứa `chips pretzels`. Lift ≈ 3.06 cho thấy khả năng xuất hiện `chips pretzels` khi hai sản phẩm ở vế trái xuất hiện cao hơn khoảng 3 lần so với mức kỳ vọng nếu các mặt hàng độc lập.

**Ý nghĩa kỹ thuật:** luật có mức liên kết tương đối mạnh, mặc dù Support thấp do tập hợp sản phẩm này xuất hiện trong một tỷ lệ nhỏ giao dịch. Điều này minh họa rằng một luật có thể có Support thấp nhưng vẫn đáng chú ý nếu Lift cao.

### Luật 2: `canned jarred vegetables, fresh herbs → fresh vegetables`

* **Support = 0.016423**
* **Confidence = 0.926136**
* **Lift = 2.104133**

Confidence ≈ 92.6% là mức rất cao, nghĩa là phần lớn các giao dịch chứa `canned jarred vegetables` và `fresh herbs` cũng chứa `fresh vegetables`. Lift ≈ 2.10 cho thấy sự xuất hiện của `fresh vegetables` trong nhóm giao dịch này cao hơn khoảng 2.1 lần so với mức kỳ vọng khi hai tập item độc lập.

**Ý nghĩa kỹ thuật:** đây là luật có **Support tương đối lớn, Confidence rất cao và Lift > 1**, do đó đồng thời thể hiện mức độ xuất hiện đáng kể và mối liên kết dương rõ ràng trong tập dữ liệu.

### Luật 3: `milk, pasta sauce → packaged cheese`

* **Support = 0.011184**
* **Confidence = 0.573643**
* **Lift = 2.476473**

Confidence ≈ 57.4% cho thấy hơn một nửa số giao dịch chứa `milk` và `pasta sauce` cũng chứa `packaged cheese`. Lift ≈ 2.48 cho thấy khả năng xuất hiện `packaged cheese` trong trường hợp này cao hơn khoảng 2.48 lần so với trường hợp các item độc lập.

**Ý nghĩa kỹ thuật:** luật đồng thời đạt Support > 1%, Confidence > 0.5 và Lift > 2, do đó thể hiện một mối liên kết có ý nghĩa giữa các item. Đây là ví dụ về luật có sự cân bằng giữa **độ phổ biến** và **độ mạnh của quan hệ**.

### Luật 4: `Weekday_Order, granola → yogurt`

* **Support = 0.011940**
* **Confidence = 0.555035**
* **Lift = 2.101363**

Confidence ≈ 55.5% cho biết hơn một nửa các giao dịch chứa `Weekday_Order` và `granola` đồng thời chứa `yogurt`. Lift ≈ 2.10 cho thấy sự xuất hiện của `yogurt` trong điều kiện trên cao hơn khoảng 2.1 lần so với mức kỳ vọng nếu các item độc lập.

**Ý nghĩa kỹ thuật:** luật này cho thấy association rule mining có thể phát hiện không chỉ quan hệ giữa các nhóm sản phẩm mà còn giữa **thuộc tính ngữ cảnh/thời gian và sản phẩm**. Vì `Weekday_Order` được mã hóa như một item trong basket, thuật toán có thể phát hiện sự kết hợp giữa thời điểm đặt hàng trong tuần và hành vi mua sản phẩm.